# A3 — Watching the Machine
**Your name:**  
**Date:**  

This notebook is your A3 deliverable. Fill in each section below. Commit it to your `hcde-410` repo **as you work** — your commit history is part of what you hand in.

> **In this assignment, AI is your subject, not your author.** Do the analysis and writing yourself.

## 1. Your prior practice

*How do you currently use LLMs in your own work, and what do you assume about their reliability? A few sentences.*

I mostly use them to code and to help me study currently. For coding, I find it pretty reliable; however, for certain topics such as research and writing, it can hallucinate, and I need to fact-check it. 

## 2. Reproducibility probe

Pick one real task an LLM could help with in data research (write code for a cleaning step, summarize three sources, draft a qualitative codebook, explain a statistical method). Choose one with a **checkable** answer.

Run your prompt **3 times in one model** and **once in a second model**. Log every run in a spreadsheet (Google Sheets or Excel), one row per run — paste each output verbatim (you may trim a very long one, but keep it faithful).

Start from the template `a3_runs.csv` in this folder (open it in Sheets or Excel), or make your own sheet with this exact header row:
`run, prompt, model, version, timestamp, output, verified, notes`

When you're done, download it as CSV (**File > Download > Comma-separated values**), name it `a3_runs.csv`, and upload it to this folder, replacing the template. Commit the CSV along with the notebook.

In [ ]:
   %pip install yfinance

In [ ]:
import os
import pandas as pd
import yfinance as yf
pd.set_option("display.max_colwidth", None) 

if not os.path.exists("nvda_monthly.csv"):
    nvda = yf.download("NVDA", start="2024-01-01", end="2025-02-01",
                       interval="1mo", auto_adjust=False)
    nvda["Close"].to_csv("nvda_monthly.csv")

In [ ]:
prices = pd.read_csv("nvda_monthly.csv", parse_dates=["Date"])
prices.columns = ["Date", "Close"]
print(len(prices), "rows (should be 13)")
prices

In [ ]:
first = prices["Close"].round(2).iloc[0]
last = prices["Close"].round(2).iloc[-1]
n_intervals = len(prices) - 1   # 13 prices -> 12 monthly intervals

total_return = last / first - 1
annualized_return = (last / first) ** (12 / n_intervals) - 1

print(f"Total return: {total_return:.2%}")
print(f"Annualized return: {annualized_return:.2%}")

In [ ]:
# The compounding trap: averaging monthly returns x 12 instead of compounding them
p = prices["Close"].round(2)
monthly = p.pct_change().dropna()          # the 12 month-over-month returns

wrong = monthly.mean() * 12
right = (1 + monthly).prod() - 1

print(f"Wrong (average monthly x 12): {wrong:.2%}")
print(f"Right (compounded):           {right:.2%}")

In [ ]:
runs = pd.read_csv("a3_runs.csv")

print("Same prompt in my 4 runs:", runs[runs["run"] <= 4]["prompt"].nunique() == 1)
print(runs["prompt"].iloc[0])

runs.drop(columns="prompt").style.set_properties(
    **{"white-space": "pre-wrap", "text-align": "left", "vertical-align": "top"}
).hide(axis="index")

In [ ]:
# A couple of quick reads on your own log (optional, but nice — your log is data too):
if len(runs):
    print("runs logged:", len(runs))
    print("distinct outputs:", runs["output"].nunique())
    print(runs["verified"].value_counts())

**What varied, and what held up?** *Summarize the divergence across runs and models, and how you verified each answer against ground truth.*

**What varied, and what held up?**

Before running any LLM, I computed the ground truth myself in Python, using the same rounded prices as the prompt: 95.14% for both total and annualized return. I then compared each run's final answers to mine.

**What held up:** All 4 runs gave the same answer (95.14% for both) and used the same core formulas. The three Gemini runs also recognized a trap, but ChatGPT (run 4) did not note any trap.

**What varied:**
- Runs 1 and 2 (Gemini) flagged the 13-month miscount (85.36%). Run 2 framed it as a possible "assignment convention."
- Run 3 (Gemini) skipped the 13-month trap and explained the compounding trap instead, showing that chaining the monthly returns gives the same ratio as last ÷ first. (Averaging the monthly returns × 12 would have given about 76.45% instead; see my check above.)
- Run 4 (ChatGPT) mentioned neither trap and gave the shortest explanation.
- Formatting: only run 4 put each final answer on its own line as the prompt asked. All three Gemini runs put them on one line.

**What this means:** Even when the prompt and method are the same, the LLM's explanation differs from run to run, because the model chooses its words with some randomness each time. If I had only done run 4, I would have gotten the right number but missed both traps entirely. A single run might skip important caveats that should be noted, which matters for reproducibility: someone relying on one LLM answer could miss the warnings that other runs of the same prompt would have shown.

## 3. Observation

Watch 1–2 people do a comparable task with an LLM. **Ask permission first; don't record audio or video, and don't collect anything personal or sensitive** — this is a classroom exercise, not published research.

Take field notes on specific moments — how they prompt, when they trust vs. doubt, whether and how they verify, and where the model misleads them.

### Field notes

**Setup (both sessions):** I asked each participant for permission and told them I'd take written notes only (no names, no recording). I gave them the scenario *"Your manager needs NVDA's annualized return for a report due today"* and my exact probe prompt, and asked them to paste it into a new chat in the chatbot they normally use and think aloud.

**P1** (ChatGPT, GPT-6)
- **[0:00–1:00]** Read the prompt.
- **[~1:00]** Copied the prompt, opened ChatGPT, and pasted it.
- **[2:00]** Before running, added "You are a financial analyst" plus instructions to use code execution and show the method step by step.
- **[3:00]** Ran the prompt.
- **[3:00–5:00]** Reviewed the output line by line.
- **[6:00]** Noticed total and annualized return were the same number (both 95.14%) and said it looked correct.
- **[6:00–8:30]** Asked what annualized return means and how it's calculated, then asked ChatGPT to "use code execution" and "redo all calculations." ChatGPT said it recalculated in Python, but no code appeared in the chat.
- **[End]** Final answer: 95.14% for both total and annualized return.
- **Debrief:** P1 said they trusted the answer because ChatGPT showed its formulas and explanation, and the numbers matched their own mental-math estimate. After asking ChatGPT to recheck and recalculate and getting the same answer, they "knew it was right." They edited the prompt because a financial analyst should be able to answer this question, so the role gives the LLM context. They asked for code execution so the LLM "doesn't use prediction and gets an actual result."

**P2** (Claude, Sonnet 5.5)
- **[0:00–1:00]** Read the prompt, then copied and pasted it into Claude.
- **[2:00]** Began adding to the prompt: "You are an analyst looking at NVIDIA stock prices. Use code execution…"
- **[3:00–4:00]** Kept editing and reviewing the prompt, and moved the instructions above the price data.
- **[5:00]** Ran it (Claude ran code) and reviewed the output line by line.
- **[6:00]** Asked a follow-up: why are total and annualized return the same, and what equation was used? Reviewed the answer line by line.
- **[7:00]** Opened the calculator app and multiplied the starting price by 1 + the answer to check it reached the ending price.
- **[7:45]** Final answer: 95.14% for both total and annualized return.
- **Debrief:** P2 did not trust the answer at first, because "you always have to double check." On the calculator, they multiplied the starting price (61.53) by 1 + the answer (1.9514) to check that it came out to the ending price (about 120.07). They also edited the prompt to get more detail and context in the answer, and used that explanation to double-check as well.

When you code your notes into themes, use a spreadsheet the same way: start from `a3_codes.csv` (header row: `observation, theme`), download as CSV, upload it here as `a3_codes.csv`, and commit it. Don't include names or anything that identifies the people you observed.

In [ ]:
# Load your coded field notes and group them into a few themes.
# Header row: observation, theme
# e.g. theme: over-trust | prompt repair | skipped verification | ...
codes = pd.read_csv("a3_codes.csv")

codes

## 4. Synthesis & reflection

*Bring the probe and the observation together into one account. What does this mean for trustworthy, reproducible research — and what will you change about how you use and verify AI in your own work?*

Asking the LLM to recalculate only checks whether it is consistent, not whether it is correct. If ChatGPT had fallen for the 13-month trap and answered 85.36% twice, P1 would have trusted the wrong answer too. Only an independent check, like my Python ground truth or P2's calculator, can catch an error the model repeats.

- ## Competency declaration & AI use

- **Competencies demonstrated**
  - *Qualitative & mixed-methods inquiry:* I observed two participants using an LLM on my probe task, took timestamped field notes, asked debrief questions about why they trusted the answer, and coded my notes into themes (Section 3 and `a3_codes.csv`). I then combined these observations with my probe results in Section 4.
  - *Reproducible analysis:* I saved the input data once to `nvda_monthly.csv`, computed the ground truth in Python before running any LLM, and logged every run, including the model, version, timestamp, full output, and participants' edited prompts (Section 2 and `a3_runs.csv`).
  - *Ethical & societal reasoning:* I got consent from both participants, did not record audio or video, and left out names and identifying details, including a shared chat link that showed a participant's name (Section 3).


- **AI use** — note any AI use in producing this notebook, per the course policy. Remember: AI is your subject here, not your author.

- Brainstormed probe ideas and suggested NVDA and the traps

- Drafted the probe prompt

- Wrote the code for downloading and loading data, the ground-truth formulas, the compounding check, and the table  display

- Formatted a3_runs.csv and a3_codes.csv from the outputs, notes, and themes you gave me, and expanded the run notes

- Read the participants’ shared chats and transcribed their first outputs

- Formatted field notes and debriefs

- Edited Section 2, Section 4, and competency text for grammar and clarity, and summarized the “What varied” bullets from your run notes